# Orthomosaic — nb1b: Dense MVS (GPU)

```
  ┌──────────────────────────────────────────────────────────────────┐
  │ config_nb  ·  GPS clustering, DENSE_MAX_IMAGE_SIZE, camera/GSD     │
  └──────────────────────────────────┬─────────────────────────────────┘
                                      ▼   CPU compute
  ┌──────────────────────────────────────────────────────────────────┐
  │ nb1a  download → EXIF/QC → CPU SIFT (distributed) → matching       │
  │       → sparse reconstruction → SPARSE ORTHOMOSAIC                 │
  │       (persists the sparse model + GPS priors)                    │
  └──────────────────────────────────┬─────────────────────────────────┘
                                      │   want dense? (optional) → run nb1b
                                      ▼   GPU compute (driver, 8×H100)
  ┌──────────────────────────────────────────────────────────────────┐
  │ nb1b  load nb1a's sparse model → undistort → recommend_dense_      │
  │       allocation → dense_mvs_pool (GPU patch_match / cluster)      │
  │       → stereo_fusion → georef → DENSE ORTHOMOSAIC / DSM / LAZ     │
  └──────────────────────────────────┬─────────────────────────────────┘
                                      ▼   CPU · ortho_input = dense if present, else sparse
  ┌──────────────────────────────────────────────────────────────────┐
  │ 02_publish  color correction → COG → PMTiles                       │
  └──────────────────────────────────────────────────────────────────┘
```

◀ **you are here** — nb1b (dense, GPU)

Running this notebook opts into **dense** reconstruction on **GPU compute**. It loads nb1a's persisted sparse model, undistorts each GPS cluster, runs COLMAP `patch_match_stereo` across the node's GPUs via GeoBrix `dense_mvs_pool` (a driver-side GPU-slot scheduler), fuses the depth maps, and writes a dense RGB ortho + DSM + LAZ. `GPU = True` (set below, before `%run ./config_nb`) is what tells `config_nb` this is the GPU run.

```
  Driver (GPU node): N × GPU · shared host RAM · cores
    rx.recommend_dense_allocation(cluster_sizes, dense_max_image_size, rx.gpu_infra())
          |  -> { concurrency, gpu_index per slot, cache_size }
          v   rx.dense_mvs_pool   (driver-side slot scheduler)
    +--------------------------------------------------------------+
    | queue: [cluster0, cluster1, ..., clusterN]                   |
    |   GPU0   GPU1   GPU2   ...   (up to `concurrency` at once)    |
    |   [c0]   [c1]   [c2]   ...   <- patch_match_stereo / cluster  |
    |   on completion -> pull the next cluster onto the freed GPU  |
    |   fewer clusters than GPUs -> each task gets several (NVLink) |
    +--------------------------------------------------------------+
```

> **Runtime.** Serverless GPU AI Runtime env5 (pycolmap-cuda12, CUDA 12). The CPU peer is nb1a (01a_sfm_orthomosaic), which produces the sparse orthomosaic + the persisted sparse model this notebook loads.

> **Recommended environment: Databricks AI Runtime on a GPU cluster — `GPU_8xH100` (or comparable).**
>
> Dense MVS (`patch_match_stereo`) is **CUDA-only, GPU-bound, and memory-hungry** — it needs a **beefy multi-GPU node**, not a laptop-class GPU. On **Serverless**, request the **GPU AI Runtime** (`--hardware-accelerator GPU_8xH100`, env 5 = CUDA 12, matching `pycolmap-cuda12`); on a **classic** cluster use a large-RAM multi-GPU node. A small / single-GPU node is slow and can exhaust host RAM. GeoBrix's `recommend_dense_allocation` fans the sparse clusters across **all available GPUs** automatically — more GPUs → more clusters densified in parallel.

## Setup

In [ ]:
# Install GeoBrix (GPU pycolmap-cuda12) for INTERACTIVE runs. Job runs strip this cell and inject
# deps from the environment spec (--extras photogrammetry_gpu_env5) instead — interactive-only.
# NOTE: light_dbr19 = classic DBR 18/19 interactive (protobuf-6). On a Serverless env5
# interactive session, swap to light_env5. Job runs strip %pip (deps come via --extras),
# so this line only affects interactive attach.
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall "geobrix[light_dbr19,photogrammetry_gpu_env5,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet "geobrix[light_dbr19,photogrammetry_gpu_env5,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%restart_python

In [ ]:
%run ./config_nb

### Optional: a faster “taste” of dense MVS (opt-in)

By **default this notebook densifies the FULL sparse reconstruction** from nb1a — the full orthomosaic. For a quick, lower-fidelity preview (much faster, and runnable on a smaller GPU), uncomment the settings in the next cell: one cluster and its first 8 images (dense keeps the full 1600 px resolution — the speed-up is the smaller image set, not downscaling). **Pair with nb1a's matching opt-in** so its sparse model is built at the same scale. Leave all three commented for the full run.

In [ ]:
# --- Optional dev "taste": a MINIMAL dense reconstruction instead of the full one ---
# Uncomment to preview quickly; leave commented (default) for the FULL run. Pair with the
# matching opt-in in nb1a so its sparse model is built at the same 8-image scale.
# DEV_MAX_IMAGES = 8          # densify only this cluster's first 8 images (needs nb1a's dev model)
# SUBSET_CLUSTER = 0          # densify only GPS cluster 0
# DENSE_MAX_IMAGE_SIZE = 1600  # keep full res; 800 starved the cloud (no usable dense)

In [ ]:
# nb1b requires GPU compute (dense patch_match needs CUDA). Assert the driver sees a GPU.
from databricks.labs.gbx import pyrx as rx
_infra = rx.gpu_infra()
assert _infra["gpu_count"] >= 1, (
    "nb1b needs GPU compute (0 CUDA devices on the driver). Run on Serverless GPU AI Runtime, "
    "or run nb1a alone for the sparse orthomosaic."
)
print(f"[nb1b] GPU: {_infra['gpu_count']}x, {_infra.get('per_gpu_vram_mb',0)//1024}GB VRAM each, "
      f"{_infra.get('host_ram_mb',0)//1024}GB host RAM, {_infra.get('cores',0)} cores "
      f"(driver {_infra.get('driver','?')}, CUDA {_infra.get('cuda','?')})")

> **Compute note.** Sparse SfM runs **CPU-distributed** (per-image extraction + per-pair matching as Spark tasks) even on this GPU node — per-task CUDA init on a shared GPU makes GPU sparse counterproductive here. The **GPU is used for the dense MVS stage** (`patch_match_stereo`), a single per-cluster context. Dense lands in a following step.

## Stage the image dataset (idempotent)

Download the [Old Orchard DroneDB dataset](https://hub.dronedb.app/r/odm/old-orchard) from
GitHub. The `FORCE_DOWNLOAD=False` guard skips the fetch when images are already staged.

In [ ]:
import time as _time_import  # time imported in config_nb; alias for clarity

img_path = str(Path(DOWNLOAD_DIR).resolve())

# Per-file staging: always fetch the authoritative GitHub listing, then download
# only the images MISSING locally (fill gaps in a partial download) instead of
# skipping wholesale when >=1 file is already present. FORCE_DOWNLOAD=True
# re-fetches every file even if present; otherwise each file already on disk is
# kept. (Lists the repo every run via the cheap contents API so an incomplete
# local set is completed rather than skipped.)
_t = _time_import.perf_counter()
image_files = [f for f in iter_repo_files(GH_DATASET_PATH) if is_image(f["path"])]
_fetched = _kept = 0
for fi in image_files:
    _out = Path(DOWNLOAD_DIR) / Path(fi["path"]).relative_to(GH_DATASET_PATH)
    if _out.exists() and not FORCE_DOWNLOAD:
        _kept += 1
        continue
    download_file(fi, DOWNLOAD_DIR, force=FORCE_DOWNLOAD)
    _fetched += 1
staged = [p for p in Path(DOWNLOAD_DIR).rglob("*") if p.suffix.lower() in (".jpg", ".jpeg")]
print(f"{len(image_files)} source images: {_fetched} fetched "
      f"({'FORCE_DOWNLOAD' if FORCE_DOWNLOAD else 'missing-only'}), {_kept} kept; "
      f"{len(staged)} staged at {img_path} in {_time_import.perf_counter()-_t:.1f}s")

## Preview: the raw input images (the “before”)

Before any processing, the input is just a pile of **overlapping drone photos** — each
carries per-image GPS but has no alignment to its neighbors or to the ground. The contact
sheet below samples the raw frames so the challenge is visible: Structure-from-Motion must
recover every camera’s pose and a sparse 3-D scene, which we then back-project into a
single georeferenced orthomosaic.

In [ ]:
# Contact sheet of the raw inputs: an evenly-spaced sample of the overlapping,
# GPS-only drone frames that SfM must align and stitch. vz.plot_gallery previews
# any image/raster collection as a thumbnail grid (the small-multiples of plot_file).
vz.plot_gallery(
    img_path,
    mode="sample",
    limit=24,
    cols=6,
    title="Raw input drone images (unstitched, GPS-only)",
)

## Extract EXIF + GPS metadata with GeoBrix `exif_gbx`

**GeoBrix `exif_gbx`** — a Serverless-safe DataSource V2 reader — extracts per-file EXIF
metadata, GPS coordinates (as WKB geometry), focal length, sensor width, and image
dimensions without hand-rolling PIL loops.

`mode="qc"` additionally computes `sharpness` and `brightness` pixel metrics in the same
pass (no second image decode), saving one full round-trip over the file set.

In [ ]:
import time as _t_exif

_t0 = _t_exif.perf_counter()
try:
    # Auto-detect the collection's COMMON EXIF intrinsics (uniform focal + a
    # camera-model sensor-width lookup) and surface an evaluation of what is
    # common / present-but-variable / missing. Explicit config still wins.
    from databricks.labs.gbx.ds.exif import build_exif_common_opts
    _exif_opts, _exif_report = build_exif_common_opts(img_path, mode="sample", limit=24)
    _exif_opts["mode"] = "qc"
    if SENSOR_WIDTH_MM is not None:
        _exif_opts["sensorWidthMm"] = str(SENSOR_WIDTH_MM)
    if FOCAL_LENGTH_MM is not None:
        _exif_opts["focalLengthMm"] = str(FOCAL_LENGTH_MM)
    print("[exif] common opts:", _exif_opts)
    print("[exif] classification:", _exif_report["classification"])
    for _s in _exif_report["suggestions"]:
        print("   [exif] suggestion:", _s)

    # spark.read.format("exif_gbx") — one row per image; no PIL loop needed.
    df_meta = (
        spark.read.format("exif_gbx")
             .options(**_exif_opts)
             .load(img_path)
    )

    # Add group key: default constant group so the pipeline runs as one batch.
    # Change GROUP_KEY_COL and assign a real column here for multi-group runs.
    if GROUP_KEY_COL == "_group":
        df_meta = df_meta.withColumn("_group", F.lit("all"))
    # else: GROUP_KEY_COL already exists in the schema (e.g. derived from timestamp)

    print(f"exif_gbx: {df_meta.count()} images read in {_t_exif.perf_counter()-_t0:.1f}s")
    df_meta.select(
        "path", "latitude", "longitude", "altitude",
        "focal_length_mm", "sensor_width_mm", "image_width", "image_height",
        "sharpness", "brightness",
    ).show(5, truncate=60)
except Exception as e:
    print(f"[ERROR] exif_gbx step failed after {_t_exif.perf_counter()-_t0:.1f}s: {e}")
    raise

## Quality-control filter

Remove blurry or underexposed images before SfM.

- **Sharpness** — keep images with sharpness ≥ 50% of the batch median.
- **Brightness** — keep images in the range [20, 240] (discard over- and under-exposed).

The `sharpness` and `brightness` columns were populated by `exif_gbx` mode `"qc"` using the
same `image_sharpness` / `image_brightness` scalars from `databricks.labs.gbx.pyrx.imagery`.

In [ ]:
import time as _t_qc

_t0 = _t_qc.perf_counter()
try:
    window_spec = Window.partitionBy(GROUP_KEY_COL)
    df_qc = (
        df_meta
        .withColumn("median_sharpness", F.percentile_approx("sharpness", 0.5).over(window_spec))
        .filter(F.col("sharpness") >= F.col("median_sharpness") * 0.5)
        .filter((F.col("brightness") > 20) & (F.col("brightness") < 240))
        .withColumn("gps_geom", F.expr("st_point(longitude, latitude)"))
        .withColumnRenamed("path", "source")
    )
    n_before = df_meta.count()
    n_after  = df_qc.count()
    print(f"QC filter: {n_before} → {n_after} images retained "
          f"({n_before - n_after} removed) in {_t_qc.perf_counter()-_t0:.1f}s")
except Exception as e:
    print(f"[ERROR] QC filter failed after {_t_qc.perf_counter()-_t0:.1f}s: {e}")
    raise

## Validate GPS coverage

Warn before SfM if fewer than 3 images have valid GPS — incremental mapping will proceed
but without GPS-constrained bundle adjustment, which reduces georeferencing accuracy.

In [ ]:
gps_ok_count = df_qc.filter(
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull() & F.col("altitude").isNotNull()
).count()
total_count = df_qc.count()
print(f"GPS coverage: {gps_ok_count}/{total_count} images have valid lat/lon/altitude")
if gps_ok_count < 3:
    print("[WARN] Fewer than 3 images have GPS — incremental mapping will proceed without "
          "GPS-constrained BA. Georeferencing accuracy may be reduced.")

## Run distributed SfM (one orthomosaic per `GROUP_KEY_COL` value)

For each distinct value of `GROUP_KEY_COL`, collect the image subset and run
`run_spark_sfm` — feature extraction, geospatial pair discovery, distributed matching,
and incremental mapping via `pycolmap`.

The `GROUP_KEY_COL = "_group"` default runs all images as a single group.
Set `GROUP_KEY_COL` to a metadata column (e.g. `"flight_date"`) and pass a real column
value to run the pipeline for each flight.

In [ ]:
# nb1b LOADS nb1a's PERSISTED sparse model directly — it does NOT run SfM extraction.
# nb1b installs pycolmap-cuda12 (for GPU dense); the distributed CPU-SIFT extraction cannot
# import the CUDA build on GPU-less mapInPandas executors, so extraction is nb1a's job. Here we
# just read the persisted sparse model + GPS priors that nb1a wrote to the Volume.
from pathlib import Path as _Path
groups = [row[GROUP_KEY_COL] for row in df_qc.select(GROUP_KEY_COL).distinct().collect()]
cluster_models = {}
for grp in groups:
    _proot = _Path(group_paths(grp)["sfm_persist"])
    cluster_models[grp] = {}
    _cands = {}
    for _cdir in sorted(_proot.glob("cluster_*")):
        _sp = _cdir / "sparse"
        _gj = _cdir / "gps_priors.json"
        if not (_sp.exists() and _gj.exists()):
            continue
        _nm = _cdir.name[len("cluster_"):]
        _cid = int(_nm.split("_")[0]) if _nm.split("_")[0].isdigit() else _nm
        _cands.setdefault(_cid, []).append(("_dev" in _nm, str(_sp), str(_gj)))
    # Prefer canonical cluster_<cid> over a cluster_<cid>_dev<N> artifact when both exist,
    # so a full run never picks up a small dev model; an isolated dev dir has only the
    # _dev model, so it is still used there.
    for _cid, _lst in _cands.items():
        _lst.sort(key=lambda t: t[0])
        cluster_models[grp][_cid] = (_lst[0][1], _lst[0][2])
    # Dev: SUBSET_CLUSTER=<idx> densifies only that GPS cluster (fast dev loop; mirrors
    # nb1a's SUBSET_CLUSTER). Applied per group before dense so undistort/patch_match
    # only touch one cluster.
    _subset = globals().get("SUBSET_CLUSTER", None)
    if _subset is not None and cluster_models[grp]:
        _cids = sorted(cluster_models[grp])
        if int(_subset) < len(_cids):
            _keep = _cids[int(_subset)]
            cluster_models[grp] = {_keep: cluster_models[grp][_keep]}
            print(f"[nb1b][dev] SUBSET_CLUSTER={_subset}: group {grp!r} \u2192 only cluster {_keep}")
    print(f"[nb1b] group {grp!r}: loaded {len(cluster_models[grp])} persisted sparse cluster(s) from {_proot}")
    if not cluster_models[grp]:
        raise RuntimeError(
            f"nb1b: no persisted sparse model for group {grp!r} at {_proot}. "
            "Run nb1a first — it builds + persists the sparse model this notebook consumes."
        )

## Compute GSD via GeoBrix `gsd_from_telemetry`

**GeoBrix `gsd_from_telemetry`** computes ground sampling distance in cm/pixel from
altitude (m), focal length (mm), sensor width (mm), and image width (px) using the
standard pinhole-camera formula:

```
GSD (cm/px) = sensor_mm × alt_m × 100 / (focal_mm × width_px)
```

This matches the formula used by OpenDroneMap and COLMAP documentation.

In [ ]:
import time as _t_gsd

_t0 = _t_gsd.perf_counter()
try:
    # Use exif_gbx columns (sensor_width_mm, focal_length_mm) + config override fallbacks.
    _sw_col = F.coalesce(F.col("sensor_width_mm"), F.lit(SENSOR_WIDTH_MM))
    _fl_col = F.col("focal_length_mm") if FOCAL_LENGTH_MM is None else F.lit(FOCAL_LENGTH_MM)

    df_gsd = df_qc.withColumn(
        "gsd_cm",
        gsd_udf(
            F.col("altitude").cast("double"),
            _fl_col.cast("double"),
            _sw_col.cast("double"),
            F.col("image_width").cast("double"),
        ),
    )
    stats   = df_gsd.agg(F.avg("gsd_cm"), F.min("gsd_cm"), F.max("gsd_cm")).collect()[0]
    avg_gsd = stats[0]
    print(f"GSD: avg={avg_gsd:.2f} cm/px  min={stats[1]:.2f}  max={stats[2]:.2f}  "
          f"({_t_gsd.perf_counter()-_t0:.1f}s)")

    # Use config GSD_CM if set; otherwise auto-derive from image telemetry.
    gsd_for_ortho = GSD_CM if GSD_CM is not None else round(avg_gsd, 1)
    print(f"Using GSD_CM={gsd_for_ortho:.1f} cm/px for orthomosaic rendering.")
except Exception as e:
    print(f"[ERROR] GSD step failed after {_t_gsd.perf_counter()-_t0:.1f}s: {e}")
    raise

## Dense MVS (GPU) → dense ortho + DSM + LAS/LAZ

For each reconstructed cluster, undistort images against the sparse model (`rx.dense_undistort`), run COLMAP `patch_match_stereo` across the node's GPUs via `rx.dense_mvs_pool` (driver-side GPU-slot scheduler), and fuse depth maps (`rx.dense_fuse` → `fused.ply`). Then **every cluster is placed in ONE shared ENU frame** — the anchor cluster georeferenced to GPS, neighbours tied through their shared overlap cameras (the same frame the sparse `accumulate_orthomosaic` uses) — and `dense_clusters_to_products` writes:

- a **per-cluster** dense RGB ortho + DSM, named `…_<group>_<cluster>` (e.g. `orthomosaic_dense_all_0.tif`, `dsm_dense_all_0.tif`), one set per GPS cluster;
- **sharded LAZ parts** written via the `lidar_gbx` DataSource writer (phase 1): `*_<group>_<cluster>.laz` parts, one shard per cluster, all under the group's dense output directory; and
- a **merged `dense_merged.laz`** (phase 2): all cluster parts folded into one cloud with `keepParts=True` so the individual shards are retained. Also: a **single co-registered merged** `orthomosaic_dense.tif` + `dsm_dense.tif` — the group deliverable the downstream notebooks consume.

`rx.recommend_dense_allocation` auto-tunes patch_match concurrency (GPU count, VRAM, image resolution) before the pool is launched. Dev can restrict to a single GPS cluster (`SUBSET_CLUSTER`); the full survey densifies **and merges every cluster** (recommended on `GPU_8xH100`).

> **Checkpoint.** Dense is checkpointed per cluster — each cluster's `fused.ply` is persisted to the Volume; a re-run skips completed clusters and re-does only the (CPU) georeference + merge (set `FORCE_DENSE=True` to recompute).

In [ ]:
import time as _t_dense
from databricks.labs.gbx import pyrx as rx

# Dense MVS resolution + speed knobs. geom_consistency=False is the primary, SAFE
# fast lever (~2x: skips the geometric refinement pass) WITHOUT starving the cloud.
# The src-images / iterations / window-step knobs default to COLMAP quality — capping
# ALL of them together produced a degenerate cloud (empty ground band). --set-var to
# tune per run. globals().get keeps runner-injected overrides.
DENSE_MAX_IMAGE_SIZE = globals().get("DENSE_MAX_IMAGE_SIZE", 1600)
DENSE_SRC_IMAGES     = globals().get("DENSE_SRC_IMAGES", None)            # source imgs/reference (None=all)
DENSE_GEOM_CONSIST   = bool(globals().get("DENSE_GEOM_CONSIST", False))   # 2nd (geometric) pass ~2x; OFF=fast+safe
DENSE_ITERS          = globals().get("DENSE_ITERS", None)                 # None = COLMAP default (5)
DENSE_WINDOW_STEP    = globals().get("DENSE_WINDOW_STEP", None)           # None = COLMAP default (1)

_infra = rx.gpu_infra()
print(f"[dense] GPU {_infra['gpu_count']}x (driver {_infra.get('driver','?')}, CUDA {_infra.get('cuda','?')}) | "
      f"knobs: max_image_size={DENSE_MAX_IMAGE_SIZE}, src_images={DENSE_SRC_IMAGES}, "
      f"geom_consistency={DENSE_GEOM_CONSIST}, iters={DENSE_ITERS}, window_step={DENSE_WINDOW_STEP}")

from databricks.labs.gbx.pyrx import checkpoint as _ckpt  # noqa: F401  (rx re-exports too)

def _dense_sig(cid, sparse_dir):
    """Signature for a cluster's dense output: sparse model identity + dense knobs."""
    _bin = Path(sparse_dir) / "images.bin"
    _sz = _bin.stat().st_size if _bin.exists() else 0
    return rx.input_signature(
        {"cid": str(cid), "sparse_bin_size": _sz},
        {"max_image_size": DENSE_MAX_IMAGE_SIZE, "src_images": DENSE_SRC_IMAGES,
         "geom_consistency": DENSE_GEOM_CONSIST, "iters": DENSE_ITERS,
         "window_step": DENSE_WINDOW_STEP, "max_images": globals().get("DEV_MAX_IMAGES")})

_t0 = _t_dense.perf_counter()
_t_und = _t_dense.perf_counter()
_todo, _done, _all_specs = [], [], []
for grp in groups:
    cmods = cluster_models.get(grp, {})
    if not cmods:
        print(f"[WARN] group {grp!r}: no cluster models — skipping dense")
        continue
    _gp = group_paths(grp)
    _mani = rx.Manifest(_gp["checkpoint"])
    _dev_n = globals().get("DEV_MAX_IMAGES", None)
    _devsuf = f"_dev{int(_dev_n)}" if _dev_n else ""
    for cid, (sparse_dir, gps_json) in cmods.items():
        _work = f'{_gp["sfm_dir"]}/dense_c{cid}{_devsuf}'
        _ply_vol = f'{_gp["sfm_persist"]}/dense/cluster_{cid}{_devsuf}/fused.ply'
        _sig = _dense_sig(cid, sparse_dir)
        _spec = {"cluster_id": f"{grp}::{cid}", "grp": grp, "cid": cid,
                 "sparse_dir": sparse_dir, "gps_json": gps_json, "work_dir": _work,
                 "ply_vol": _ply_vol, "sig": _sig, "mani": _mani,
                 "max_image_size": DENSE_MAX_IMAGE_SIZE, "geom_consistency": DENSE_GEOM_CONSIST,
                 "num_iterations": DENSE_ITERS, "window_step": DENSE_WINDOW_STEP}
        _all_specs.append(_spec)
        if rx.checkpoint_skip(_mani, "dense", _spec["cluster_id"], _sig, force=bool(FORCE_DENSE)):
            print(f"[dense][skip] cluster {_spec['cluster_id']} — checkpointed ({_ply_vol})")
            _done.append(_spec)
        else:
            import shutil as _sh0
            _sh0.rmtree(_work, ignore_errors=True)  # warm cluster: /tmp persists across runs; clear stale dense workspace or patch_match hits a resolution/dep mismatch
            rx.dense_undistort(sparse_dir, img_path, _work, num_src_images=DENSE_SRC_IMAGES)  # NO max_images: subsetting breaks patch_match covisibility (dev cap belongs in nb1a)
            _todo.append(_spec)
print(f"[dense] undistort {len(_todo)} todo (+{len(_done)} checkpointed): {_t_dense.perf_counter()-_t_und:.0f}s")

dense_paths, dense_clouds, _timing = {}, {}, {}
if not _all_specs:
    print("No cluster models to densify (run nb1a first to build + persist the sparse model).")
else:
    _t_pm = _t_dense.perf_counter()
    _pool = {}
    if _todo:
        _alloc = rx.recommend_dense_allocation([1] * len(_todo), dense_max_image_size=DENSE_MAX_IMAGE_SIZE)
        print(f"[dense] {_alloc['reason']}")
        _cache_override = globals().get("DENSE_CACHE_GB")  # --set-var DENSE_CACHE_GB=N to force a tiny host cache
        for _s in _todo:
            _s["cache_size_gb"] = float(_cache_override) if _cache_override else _alloc["cache_size_gb"]
        _pool = rx.dense_mvs_pool(_todo, allocation=_alloc)
    _timing["patch_match_s"] = round(_t_dense.perf_counter() - _t_pm, 1)
    print(f"[dense] patch_match (GPU pool, {len(_todo)} clusters): {_timing['patch_match_s']}s")
    # Fuse todo clusters (persist fused.ply + checkpoint); collect per-cluster plys.
    _t_fx = _t_dense.perf_counter()
    _plys = {}  # (grp, cid) -> fused.ply path
    for spec in _all_specs:
        if spec in _done:
            _plys[(spec["grp"], spec["cid"])] = spec["ply_vol"]  # persisted — skip fuse
            continue
        _st = _pool.get(spec["cluster_id"], {})
        if _st.get("status") != "ok":
            print(f"  [DROP] cluster {spec['cluster_id']} patch_match failed: {str(_st.get('error',''))[:160]}")
            continue
        _local_ply = rx.dense_fuse(spec["work_dir"], f'{spec["work_dir"]}/fused.ply')
        Path(spec["ply_vol"]).parent.mkdir(parents=True, exist_ok=True)
        import shutil as _sh
        _sh.copy(_local_ply, spec["ply_vol"])            # persist to Volume for recovery
        spec["mani"].mark_done("dense", spec["cluster_id"], spec["sig"], spec["ply_vol"],
                               grp=spec["grp"], cid=spec["cid"])
        _plys[(spec["grp"], spec["cid"])] = spec["ply_vol"]

    # Georeference every cluster in ONE shared ENU frame, then write per-cluster
    # (_<grp>_<cid>) ortho/DSM/LAZ AND a single co-registered merged set per group
    # (mirrors the sparse accumulate_orthomosaic frame, so dense lines up with sparse).
    for grp in groups:
        _cmods = cluster_models.get(grp, {})
        _dplys = {cid: _plys[(grp, cid)] for cid in _cmods if (grp, cid) in _plys}
        if not _dplys:
            print(f"[dense] group {grp!r}: no fused clouds — skipping georef")
            continue
        _gp = group_paths(grp)
        _cpaths = {cid: dense_cluster_paths(grp, cid) for cid in _dplys}
        _used, _merged_laz = dense_clusters_to_products(
            {cid: _cmods[cid] for cid in _dplys}, _dplys,
            merged_ortho=_gp["dense_ortho"], merged_dsm=_gp["dense_dsm"],
            merged_laz=_gp["dense_cloud"], cluster_paths=_cpaths, gsd_cm=gsd_for_ortho)
        dense_clouds[grp] = _merged_laz
        dense_paths[grp] = _gp["dense_ortho"]
        print(f"[dense] group {grp!r}: {len(_used)} per-cluster + 1 merged set written")
    _timing["fuse_export_s"] = round(_t_dense.perf_counter() - _t_fx, 1)
    print(f"[dense] fuse+georef+export: {_timing['fuse_export_s']}s")
_timing["total_s"] = round(_t_dense.perf_counter() - _t0, 1)
_timing["knobs"] = {"max_image_size": DENSE_MAX_IMAGE_SIZE, "src_images": DENSE_SRC_IMAGES,
                    "geom_consistency": DENSE_GEOM_CONSIST, "iters": DENSE_ITERS,
                    "window_step": DENSE_WINDOW_STEP, "gpu_count": _infra.get("gpu_count"),
                    "skipped": len(_done), "computed": len(_todo)}
print(f"[dense] TOTAL {_timing['total_s']:.0f}s → {list(dense_paths.values())}  timing={_timing}")
try:
    import json as _json, os as _os
    _os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/_dev_dense_timing.json", "w") as _f:
        _json.dump(_timing, _f)
    print(f"[dense] wrote timing → {output_dir}/_dev_dense_timing.json")
except Exception as _e:
    print(f"[dense] timing persist skipped: {_e}")

In [ ]:
# Render the dense ortho (sharper than the sparse flat-plane back-projection).
if dense_paths:
    vz.plot_file(list(dense_paths.values())[0])

## Verify the dense cloud (lidar_gbx)

Read the exported LAZ back through the `lidar_gbx` metadata reader to confirm the merged point cloud (`dense_merged.laz`) round-trips — a non-zero `point_count` is the check that the two-phase dense export is well-formed and consumable by the LiDAR raster functions. (Per-cluster sharded parts are kept under the same directory and also readable via `lidar_gbx` as a directory of parts.)

In [ ]:
# Verify the exported dense point cloud(s) round-trip through lidar_gbx — the
# MERGED cloud AND each per-cluster cloud, so every export is confirmed well-formed
# and consumable by the LiDAR raster functions (a non-zero point_count is the check).
from pathlib import Path as _P

_to_verify = []
for _grp, _laz in dense_clouds.items():
    _to_verify.append((f"{_grp} (merged)", _laz))
    for _cid in sorted(cluster_models.get(_grp, {})):
        _cp = dense_cluster_paths(_grp, _cid)["cloud"]
        # write_xyzrgb_laz may fall back .laz -> .las; take whichever exists.
        for _cand in (_cp, str(_P(_cp).with_suffix(".las"))):
            if _P(_cand).exists():
                _to_verify.append((f"{_grp}::cluster {_cid}", _cand))
                break
for _label, _laz in _to_verify:
    try:
        _meta = spark.read.format("lidar_gbx").option("mode", "metadata").load(_laz)
        _row = _meta.select(
            "point_count", "x_min", "x_max", "z_min", "z_max", "crs"
        ).collect()[0]
        print(f"lidar_gbx: {_label} → point_count={_row['point_count']:,} "
              f"z=[{_row['z_min']:.1f}, {_row['z_max']:.1f}] crs={_row['crs']}")
    except Exception as _e:  # noqa: BLE001
        print(f"[WARN] lidar_gbx read of {_laz} failed: {_e}")

## Visualise — per-cluster dense tiles

The merged dense orthomosaic is rendered above; here is each GPS cluster's dense ortho on its own — confirming every cluster densified before they are blended into the single merged product.

In [ ]:
# Per-cluster dense orthos: a gallery confirming EVERY GPS cluster was densified
# (the merged ortho rendered above blends them into one seamless product). Falls
# back to the sparse ortho only if no dense product exists this run.
from pathlib import Path as _P

_tiles, _labels = [], []
for grp in groups:
    for cid in sorted(cluster_models.get(grp, {})):
        _op = dense_cluster_paths(grp, cid)["ortho"]
        if _P(_op).exists():
            _tiles.append(_op)
            _labels.append(f"{grp}::c{cid}")
if _tiles:
    vz.plot_gallery(
        _tiles, labels=_labels, renderer="raster",
        cols=min(4, len(_tiles)), title="Per-cluster dense orthomosaics",
    )
else:
    _shown = [g for g in groups if _P(group_paths(g)["ortho"]).exists()]
    if _shown:
        vz.plot_file(
            group_paths(_shown[0])["ortho"],
            title="Sparse orthomosaic (no dense product this run)",
        )
    else:
        print("[skip] no orthomosaic produced this run (no cluster reconstructed)")

## Steps performed

1. **Download** — fetched the Old Orchard drone imagery from GitHub (idempotent).
2. **EXIF/GPS extraction** — `exif_gbx` (GeoBrix) read metadata + sharpness/brightness in one pass.
3. **QC filter** — retained images above the per-group sharpness p50 × 0.5 and within the brightness window.
4. **GPS validation** — warned when fewer than 3 images have valid altitude.
5. **Distributed SfM** — feature extraction, `ST_DistanceSphere` (Databricks) pair discovery, distributed matching, and incremental mapping via `pycolmap`.
6. **GSD** — computed via `gsd_from_telemetry` (GeoBrix) from altitude, focal length, sensor size, and image width.
7. **Dense MVS (GPU)** — densified each cluster with COLMAP `patch_match_stereo`, placed all clusters in one shared ENU frame, and wrote per-cluster ortho + DSM plus a single merged ortho + DSM. LAZ export used the `lidar_gbx` DataSource writer in two phases: phase 1 = sharded `*_<group>_<cluster>.laz` parts (one per cluster); phase 2 = `dense_merged.laz` merged from all parts (keepParts=True so shards are retained).

**Next:** Part 2 applies per-channel percentile color correction to remove sensor cast.